# 02 — Pitchfork Bifurcation Analysis

Compute the analytical $c_1, c_3$ Taylor coefficients on a $(\kappa, \delta)$ grid; locate the pitchfork as the $c_1=0$ contour; classify supercritical vs subcritical via $\mathrm{sgn}(c_3)$; overlay the numerical fixed-point branches; verify the broad-bump anchor $\sin\Delta\cos\delta = 0$ in the $\kappa\to 0$ limit.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from steering.params import ModelParams, ForcingParams
from steering.models import (
    DuffingModel,
    BesselSteeringModel,
    ContinuousPFLModel,
    DiscretePFLModel,
    FullCircuitModel,
)
from steering.dynamics import VelocityDynamics, AccelerationDynamics
from steering.integrator import Simulation
from steering.visualization.style import use_paper_style

use_paper_style()


In [ ]:
from steering.analysis.bifurcation import (
    pitchfork_bifurcation_curve,
    numerical_bifurcation_diagram_1d,
)
from steering.visualization.bifurcation_plot import plot_bifurcation_2d

bessel = BesselSteeringModel()


## Bifurcation diagram on $(\kappa, \delta)$

We sweep $\kappa_h$ over $(\kappa_g$ slaved to it via post-processing), and $\delta$ over $[0, \pi/2]$.

In [ ]:
kappas = np.linspace(0.5, 6.0, 61)
deltas = np.linspace(0.0, np.pi/2 - 0.001, 61)
base = ModelParams(kappa_h=1.0, kappa_g=1.0, delta=0.0)
c1_grid = np.empty((kappas.size, deltas.size))
c3_grid = np.empty_like(c1_grid)
for i, k in enumerate(kappas):
    for j, d in enumerate(deltas):
        p = base.replace(kappa_h=k, kappa_g=k, delta=d)
        c1, c3 = bessel.taylor_coefficients(p)
        c1_grid[i, j] = c1
        c3_grid[i, j] = c3

from steering.analysis.bifurcation import BifurcationDiagram
diag = BifurcationDiagram(
    'kappa', 'delta', kappas, deltas, c1_grid, c3_grid,
)
fig, ax = plt.subplots(figsize=(7, 4.5))
plot_bifurcation_2d(diag, r'$\kappa$', r'$\delta$', ax=ax)
plt.show()


## Broad-bump limit anchor

For $\kappa \to 0$, the bifurcation condition reduces to $\sin\Delta\cos\delta = 0$, i.e. $\delta = \pi/2$ (with $\Delta = 3\pi/8 \neq 0$). Compute and verify.

In [ ]:
for k in [0.05, 0.1, 0.5, 1.0, 2.0, 5.0]:
    # Find delta where c1 = 0 by bisection.
    from scipy.optimize import brentq
    def c1_of(d, k=k):
        return bessel.taylor_coefficients(base.replace(kappa_h=k, kappa_g=k, delta=d))[0]
    try:
        d_star = brentq(c1_of, 0.5, np.pi/2 - 1e-3)
    except ValueError:
        d_star = float('nan')
    print(f'kappa = {k:.2f}: pitchfork at delta = {d_star:.4f} (pi/2 = {np.pi/2:.4f})')


## 1D bifurcation: stable/unstable branches as $\delta$ varies

Fix $\kappa = 2$ and find all fixed points of $U(\theta) = 0$ for each $\delta$. Show stable branches as solid points, unstable as open.

In [ ]:
data = numerical_bifurcation_diagram_1d(
    bessel, 'delta', np.linspace(0.5, np.pi/2 - 0.01, 80),
    base.replace(kappa_h=2.0, kappa_g=2.0),
    theta_range=(-np.pi, np.pi),
)
fig, ax = plt.subplots(figsize=(7, 4))
for d, ths, stabs in zip(data['param'], data['thetas'], data['stabilities']):
    for th, s in zip(ths, stabs):
        color = 'C0' if s == 'stable' else 'C3'
        ax.plot(d, th, '.', color=color, ms=3)
ax.set_xlabel(r'$\delta$'); ax.set_ylabel(r'$\theta^*$')
ax.set_title('1D bifurcation: stable (blue) vs unstable (red)')
plt.show()


## Critical damping for the acceleration system

$\gamma_c = 2\sqrt{|c_1|}$ at any fixed point. Above this, approach is overdamped; below, oscillatory.

In [ ]:
for d in [1.34, 1.40, 1.45, 1.50]:
    p = ModelParams(kappa_h=2.0, kappa_g=2.0, delta=d)
    c1, c3 = bessel.taylor_coefficients(p)
    if c1 > 0 and c3 < 0:
        # Stability eigenvalues at well: U' = c1 + 3 c3 theta_*^2 = -2 c1 (the well at sqrt(-c1/c3)).
        Up_at_well = -2 * c1
        gamma_c = 2 * np.sqrt(abs(Up_at_well))
        print(f'delta = {d:.2f}: gamma_c = {gamma_c:.3f}')
